 Aplicação espacial dos modelos em 2025

Aplica os seis modelos ao ano de validação:

- C7: suscetibilidade LR, SSR e combinação;
- C8: suscetibilidade RF, SSR e combinação.

In [ ]:
from pathlib import Path
import sys

sys.path.append("/code/scripts")

from seasonal_utils import apply_logistic_byblocks
from raster_utils import check_raster_alignment

In [ ]:
# ALTERAR APENAS ESTA VARIÁVEL

area = "centro"
year = 2025


susc_lr = Path(
    f"/code/data/results/{area}/C5/final/res_lri.tif"
)

susc_rf = Path(
    f"/code/data/results/{area}/C6/lri_model/class/"
    "rf_lri_base_mean_prob_1_10models.tif"
)

ssr = Path(
    f"/code/data/processed/{area}/meteo/ssr/"
    f"ssr_abs_{year}.tif"
)

validity_rasters = [
    susc_lr,
    susc_rf,
    ssr
]

scenario_models = {
    "C7": {
        "susc": [susc_lr],
        "ssr_abs": [ssr],
        "combined": [susc_lr, ssr]
    },
    "C8": {
        "susc": [susc_rf],
        "ssr_abs": [ssr],
        "combined": [susc_rf, ssr]
    }
}

print("Área:", area)
print("Ano:", year)

In [ ]:
outputs = []

for scenario, models in scenario_models.items():
    base = Path(
        f"/code/data/results/{area}/{scenario}/seasonal"
    )
    model_dir = base / "models"
    map_dir = base / "maps"
    map_dir.mkdir(parents=True, exist_ok=True)

    for model_name, features in models.items():
        model_file = model_dir / f"logit_{model_name}.joblib"

        probability = map_dir / (
            f"prob_{model_name}_{year}.tif"
        )

        classification = map_dir / (
            f"pred_{model_name}_{year}.tif"
        )

        apply_logistic_byblocks(
            model_file=model_file,
            feature_rasters=features,
            validity_rasters=validity_rasters,
            output_probability=probability,
            output_class=classification,
            threshold=0.5
        )

        outputs.append({
            "scenario": scenario,
            "model": model_name,
            "probability": probability,
            "classification": classification
        })

## Verificação dos outputs

In [ ]:
for output in outputs:
    check_raster_alignment(
        raster=str(output["probability"]),
        template=str(susc_lr)
    )

    check_raster_alignment(
        raster=str(output["classification"]),
        template=str(susc_lr)
    )